# Copernicus Marine Toolbox Data Downloading Examples

## Prerequisite

Before running this notebook, install the Python environment and authenticate with the Copernicus Marine service as described in `README.md`.

## Import Module

In [ ]:
# Copernicu Marine Service module to access and download the data
import copernicusmarine

# Modules to check and visualise the data
import xarray as xr
from matplotlib.colors import LogNorm

## 1. Downloading options

### 1.1. Subset of dataset 

https://help.marine.copernicus.eu/en/articles/8283072-copernicus-marine-toolbox-api-subset

the `subset` function is able to download a subset of the dataset defined by several factors, as longitude, latitude, datetimes, depth and etc. The dataset is directly downloaded in the mentioned output_diretory.

In [ ]:
# Define dataset parameters in a dictionary
dataset_params = {
    "dataset_id": "cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D",
    "variables": ["CHL", "DIATO"],
    "minimum_longitude": 50,
    "maximum_longitude": 90,
    "minimum_latitude": 0,
    "maximum_latitude": 25,
    "start_datetime": "2022-01-01T00:00:00",
    "end_datetime": "2022-01-01T23:59:59",
}

# Generate output filename based on parameters
output_filename = f"{dataset_params['dataset_id']}_{dataset_params['start_datetime'][:10]}_{dataset_params['end_datetime'][:10]}_{dataset_params['minimum_latitude']}_{dataset_params['maximum_latitude']}_{dataset_params['minimum_longitude']}_{dataset_params['maximum_longitude']}.nc"

# Download the subset using the dictionary parameters
copernicusmarine.subset(
    dataset_id=dataset_params["dataset_id"],
    variables=dataset_params["variables"],
    minimum_longitude=dataset_params["minimum_longitude"],
    maximum_longitude=dataset_params["maximum_longitude"],
    minimum_latitude=dataset_params["minimum_latitude"],
    maximum_latitude=dataset_params["maximum_latitude"],
    start_datetime=dataset_params["start_datetime"],
    end_datetime=dataset_params["end_datetime"],
    output_filename=output_filename,
    output_directory=f"data/{dataset_params['dataset_id']}/"
)

In [ ]:
# Loading and visualising the downloaded dataset
ds = xr.open_dataset(f"data/{dataset_params['dataset_id']}/{output_filename}")
ds.CHL.plot(norm=LogNorm(0.001,1))

### 1.2. Extract and show the metadata of dataset before downloading

https://help.marine.copernicus.eu/en/articles/8287609-copernicus-marine-toolbox-api-open-a-dataset-or-read-a-dataframe-remotely

The `open_dataset` function only extract and show the metadata of the file, the file could then be downloaded using `persist()`, `load()` or simply saving the file using `to_netcdf()` or `to_zarr()`. The dataset would also be downloaded and visualised directly using the `plot` function.

In [ ]:
# Define dataset parameters in a dictionary
dataset_params = {
    "dataset_id": "cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D",
    "minimum_longitude": -30,
    "maximum_longitude": -20,
    "minimum_latitude": 30,
    "maximum_latitude": 40,
    "start_datetime": "2022-01-13T00:00:00",
    "end_datetime": "2022-01-15T23:59:59",
    "variables": ["CHL", "DIATO"],
}

ds = copernicusmarine.open_dataset(
    dataset_id=dataset_params["dataset_id"],
    minimum_longitude=dataset_params["minimum_longitude"],
    maximum_longitude=dataset_params["maximum_longitude"],
    minimum_latitude=dataset_params["minimum_latitude"],
    maximum_latitude=dataset_params["maximum_latitude"],
    start_datetime=dataset_params["start_datetime"],
    end_datetime=dataset_params["end_datetime"],
    variables=dataset_params["variables"],
    chunk_size_limit=-1
)
ds

In [ ]:
## Load one day of data
ds_20220115 = ds.CHL.sel(time='2022-01-15').load()

## Save one day of data to a new file
ds_20220115.to_netcdf(f'data/{dataset_params["dataset_id"]}/{dataset_params["dataset_id"]}_CHL_2022-01-15.nc')

## Visualise one day of data
ds_20220115.plot(norm=LogNorm(0.01,1))

### 1.3. Get original file (Download the whole scene dataset e.g. global)

https://help.marine.copernicus.eu/en/articles/8286883-copernicus-marine-toolbox-api-get-original-files

A full scene (e.g., in case of global dataset a full daily global dataset) can be downloaded directly, without any subsetting, if the dataset are not downloadable via the pervisously mentioned methods. In this situation the name and data of desired data would be scanned using regular experessions (`regex`). A guideline for using regex are provided in the following image and link.

[https://docs.python.org/3/library/re.html](https://docs.python.org/3/library/re.html)

<img src="https://miro.medium.com/v2/resize:fit:1400/1*hjsbL45MhT2Tw5DGAYoAUg.png" style="width: 500px;"/>

In [ ]:
# Define dataset parameters in a dictionary
dataset_params = {
    "dataset_id": "cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D",
    "date_range": "*2022010[0-1]*",
    "output_directory": f"./data/{dataset_params['dataset_id']}/",
}

# Call the get function to save data
get_result_annualmean = copernicusmarine.get(
    dataset_id=dataset_params["dataset_id"],
    filter=dataset_params["date_range"],
    output_directory=dataset_params["output_directory"],
    no_directories=True
)

---

## 2. Downloading Phytoplankton Functional Types (PFT) dataset

### 2.1. Subset

In [ ]:
copernicusmarine.subset(
    dataset_id="cmems_obs-oc_glo_bgc-plankton_my_l3-olci-4km_P1D",
    variables=["CHL"],
    minimum_longitude = -50,
    maximum_longitude = -20,
    minimum_latitude = 0,
    # maximum_latitude = 30,
    start_datetime="2022-01-01T00:00:00",
    end_datetime="2022-01-10T23:59:59",
    output_filename = "CHL.nc",
    output_directory = "./copernicus-data/CHL_4km"
)

### 2.2. open_dataset

In [ ]:
ds = copernicusmarine.open_dataset(
    dataset_id="cmems_obs-oc_glo_bgc-plankton_my_l3-olci-4km_P1D",
    variables=["CHL"],
    minimum_longitude = -50, 
    maximum_longitude = -20,
    minimum_latitude = 0,
    # maximum_latitude = 30,
    start_datetime="2022-01-01T00:00:00",
    end_datetime="2022-01-10T23:59:59",
)

In [ ]:
ds.isel(time=5).CHL.plot(robust=True, norm=LogNorm())

# PFT YEARLY

In [ ]:
# Define parameters to run the extraction
dataset_id_yearly = "cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D"

# Define output storage parameters
output_directory = "/albedo/work/projects/p_phytooptics/emehdipo/PS113/CMEMS/data"

date_range = "*20190[1-5]*"

# Call the get function to save data
get_result_annualmean = copernicusmarine.get(
    dataset_id=dataset_id_yearly,
    filter = date_range,
    output_directory=output_directory,
    no_directories=True)

In [ ]:
dataset_id = "METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2"

copernicusmarine.subset(
  dataset_id=dataset_id,
  variables=["analysed_sst", "analysis_error"],
  minimum_longitude=-64,
  maximum_longitude=3,
  minimum_latitude=-50,
  maximum_latitude=52,
  start_datetime="2016-01-01T00:00:00",
  end_datetime="2019-12-31T23:59:59",
  output_filename = f"{dataset_id}_2016_2019.nc",
  output_directory = "/albedo/work/projects/p_phytooptics/emehdipo/PS113/GHRSST/"
)